<a href="https://colab.research.google.com/github/redddddsu/Adaptative-SNN-Based-Triggering/blob/main/Software_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing API/Library


In [1]:
!pip install snntorch

In [22]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt
import os
import librosa
import numpy as np
from snntorch import surrogate
import tensorflow as tf
import pandas as pd
import tensorflow_hub as hub
from sklearn.metrics import confusion_matrix, classification_report
import yaml




# Dataset

In [3]:

save_dir = "/content/drive/MyDrive/TUT_dataset"
os.makedirs(save_dir, exist_ok=True)

!wget -c -O /content/tut_source_events.zip \
"https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1"

!unzip -o /content/tut_source_events.zip -d /content/


--2026-11-02 08:08:38--  https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.40.132, 188.185.45.63, 188.185.47.192, ...
Connecting to zenodo.org (zenodo.org)|188.185.40.132|:443... connected.
HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.

Archive:  /content/tut_source_events.zip
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/107351.wav  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/107351.yaml  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144473.wav  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144473.yaml  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144474.w

In [4]:
babycry_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry"
glassbreak_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/glassbreak"
gunshot_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/gunshot"

all_paths = {
    "babycry": babycry_path,
    "glassbreak": glassbreak_path,
    "gunshot": gunshot_path
}

recordings = []

for event, path in all_paths.items():
    wav_files = [
        f for f in os.listdir(path)
        if f.lower().endswith(".wav")
    ]

    yaml_files = [
        f for f in os.listdir(path)
        if f.lower().endswith(".yaml")
    ]

    for filename in wav_files:
        recordings.append({
            "event": event,
            "wav_path": os.path.join(path, filename)
        })


# Mel Spectrogem

In [5]:
mel_data = {}

n_mels = 128
window_seconds = 0.064
hop_seconds = 0.032

for recording in recordings:
    event = recording["event"]
    wav_path = recording["wav_path"]

    audio, sr = librosa.load(
        wav_path,
        sr=None,
        mono=True
    )

    window_sample = round(sr * window_seconds)
    hop_sample = round(sr * hop_seconds)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=window_sample,
        win_length=window_sample,
        hop_length=hop_sample,
        n_mels=n_mels,
        fmin=0,
        fmax=sr / 2,
        power=2
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max
    )

    filename = os.path.basename(wav_path)

    mel_data[filename] = {
        "log_mel": log_mel,
        "sr": sr,
        "duration": len(audio) / sr,
        "event": event
    }

# Labeling Frames

In [6]:

    wav_path = recording["wav_path"]
    recording["yaml_path"] = os.path.splitext(wav_path)[0] + ".yaml"

event_to_label = {
    "babycry": 1,
    "glassbreak": 2,
    "gunshot": 3
}

for recording in recordings:
    filename = os.path.basename(recording["wav_path"])

    log_mel = mel_data[filename]["log_mel"]
    sr = mel_data[filename]["sr"]

    with open(recording["yaml_path"], "r") as f:
        annotation = yaml.unsafe_load(f)

    hops = round(sr * 0.032)

    labels = np.zeros(log_mel.shape[1], dtype=np.int64)

    for start, end in annotation["valid_segments"]:
        start_frame = int(np.floor(start * sr / hops))
        end_frame = int(np.ceil(end * sr / hops))

        start_frame = max(0, min(start_frame, len(labels)))
        end_frame = max(0, min(end_frame, len(labels)))

        labels[start_frame:end_frame] = event_to_label[
            recording["event"]
        ]

    mel_data[filename]["labels"] = labels


# SNN Frame


In [7]:
spike_grad = surrogate.fast_sigmoid()

class SNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(128, 128)
        self.lif1 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer2 = nn.Linear(128, 1)
        self.lif2 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

    def forward(self, x):
        potential1 = self.lif1.init_leaky()
        potential2 = self.lif2.init_leaky()

        current1_all = self.layer1(x)
        spikes = []

        for t in range(x.shape[0]):
            spk1, potential1 = self.lif1(
                current1_all[t], potential1
            )
            current2 = self.layer2(spk1)
            spk2, potential2 = self.lif2(
                current2, potential2
            )
            spikes.append(spk2)

        return torch.stack(spikes)





# Van Rossum Loss

In [8]:
def van_rossum_loss(predict, target, dt=0.032, tau=0.1):
    alpha = torch.exp(torch.tensor(-dt / tau, device=predict.device))

    pred_filter = []
    target_filter = []

    pred_state = torch.zeros_like(predict[0])
    target_state = torch.zeros_like(target[0])

    for t in range(predict.shape[0]):
        pred_state = alpha * pred_state + predict[t]
        target_state = alpha * target_state + target[t]

        pred_filter.append(pred_state)
        target_filter.append(target_state)

    pred_filter = torch.stack(pred_filter)
    target_filter = torch.stack(target_filter)

    loss = (dt / tau) * torch.mean((pred_filter - target_filter) ** 2)

    return loss

# Open-Close Filter (Post-processing step)

In [9]:
from scipy.ndimage import binary_closing, binary_opening

# baseline = 2
def close_open_filter(spikes, xi=2):
    spikes = np.asarray(spikes).astype(bool)
    if xi == 0:
        return spikes.astype(np.float32)

    closed = binary_closing(spikes, iterations=xi)
    opened = binary_opening(closed, iterations=xi)
    return opened.astype(np.float32)

# Training

In [10]:
import random

random.seed(42)
torch.manual_seed(42)

items = [
    (filename, data)
    for filename, data in mel_data.items()
    if "labels" in data
]

random.shuffle(items)

split = max(1, int(0.8 * len(items)))
split = min(split, len(items) - 1)

train_items = items[:split]
validation_items = items[split:]

print("Training recordings:", len(train_items))
print("Validation recordings:", len(validation_items))

Training recordings: 128
Validation recordings: 32


In [11]:
device = torch.device("cuda")

model = SNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def prepare_recording(data):
    log_mel = data["log_mel"].T
    x = np.clip((log_mel + 80.0) / 80.0, 0.0, 1.0)
    y = (data["labels"] > 0).astype(np.float32)
    x = torch.tensor(x, dtype=torch.float32, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)

    return x, y


epochs = 3

for epoch in range(epochs):
    model.train()
    random.shuffle(train_items)
    total_loss = 0.0

    for filename, data in train_items:
        x, target = prepare_recording(data)

        optimizer.zero_grad()

        prediction = model(x)
        loss = van_rossum_loss(prediction, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_items)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for filename, data in validation_items:
            x, target = prepare_recording(data)
            prediction = model(x)
            raw_spikes = (prediction.squeeze(-1) > 0.5).cpu().numpy()
            # filtered_spikes = close_open_filter(raw_spikes, xi=2)
            loss = van_rossum_loss(prediction, target)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(validation_items)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Validation Loss: {avg_val_loss:.6f}"
    )

Epoch 1/3 | Train Loss: 0.691477 | Validation Loss: 0.478735
Epoch 2/3 | Train Loss: 0.544464 | Validation Loss: 0.536069
Epoch 3/3 | Train Loss: 0.590145 | Validation Loss: 0.520643


# Eval

In [12]:
model.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for filename, data in validation_items:
        x, target = prepare_recording(data)
        prediction = model(x)

        pred = (prediction > 0).int().cpu().numpy().reshape(-1)
        true = (target > 0).int().cpu().numpy().reshape(-1)

        all_true.extend(true.tolist())
        all_pred.extend(pred.tolist())

print("\nClassification report:")
print(classification_report(
    all_true,
    all_pred,
    labels=[0, 1],
    target_names=["Background", "Event"],
    zero_division=0
))


Classification report:
              precision    recall  f1-score   support

  Background       0.92      0.72      0.81     32286
       Event       0.27      0.61      0.37      5496

    accuracy                           0.70     37782
   macro avg       0.59      0.66      0.59     37782
weighted avg       0.82      0.70      0.74     37782



# Result
(this is only from the validation, so we are able to tell what the tp/fp/tn/fn are)
In practice where sounds are new and random, this wont be able to tell.

In [13]:
tn, fp, fn, tp = confusion_matrix(
    all_true, all_pred, labels=[0, 1]
).ravel()

print("True Positive (TP):", tp)
print("False Positive (FP):", fp)
print("True Negative (TN):", tn)
print("False Negative (FN):", fn)

True Positive (TP): 3358
False Positive (FP): 9077
True Negative (TN): 23209
False Negative (FN): 2138


# YAMNet

In [19]:
!pip install tensorflow tensorflow-hub

yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [27]:
class_map_path = tf.keras.utils.get_file(
    'yamnet_class_map.csv',
    'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
)

class_map = pd.read_csv(class_map_path)

yamnet_event_classes = {
    "babycry": 20,
    "glassbreak": 435,
    "gunshot": 421
}

print(yamnet_event_classes)

{'babycry': 20, 'glassbreak': 435, 'gunshot': 421}
